In [0]:
import requests
from pyspark.sql import SparkSession # Import it for explicit use
from pyspark.sql.types import StructType, StructField
from datetime import datetime

In [0]:
def fetch_top_story_ids():
    url = "https://hacker-news.firebaseio.com/v0/topstories.json"
    response = requests.get(url)
    return response.json()

ids = fetch_top_story_ids()
print(f"Number of stories: {len(ids)}")
print(f"First 5 ID:s: {ids[:5]}")

In [0]:
def fetch_story(story_id):
    url = f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json"
    response = requests.get(url)
    return response.json()

story = fetch_story(ids[0])
print(story)
    

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_all_stories(story_ids, max_workers=20):
    stories = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_story, sid): sid for sid in story_ids}
        for future in as_completed(futures):
            result = future.result()
            if result:
                stories.append(result)
    return stories

stories = fetch_all_stories(ids)
print(f"Fetched: {len(stories)} stories")



In [0]:
from datetime import datetime, timezone

def clean_for_bronze(story):
    story.pop("kids", None)
    story["ingested_at"] = datetime.now(timezone.utc).isoformat()
    return story

In [0]:
def get_existing_ids():
    try:
        existing = spark.table("hacker_news_bronze").select("id").collect()
        return {row["id"] for row in existing}
    except Exception:
        return set()
    
existing_ids = get_existing_ids()
new_ids = [i for i in ids if i not in existing_ids]
print(f"Exists in: {len(existing_ids)}")
print(f"New to fetch: {len(new_ids)}")

In [0]:
if new_ids:
    cleaned = [clean_for_bronze(s) for s in fetch_all_stories(new_ids)]
    spark.createDataFrame(cleaned).write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("hacker_news_bronze")
    print(f"Addded {len(cleaned)} new stories")
else:
    print("No new stories")
            

In [0]:
spark.table("hacker_news_bronze").count()